# 01 — Exploração dos dados (KDD, fases 1 a 3)

Notebook **guiado**: você completa os `TODO`s e responde às perguntas. A explicação do KDD está em `docs/guia/05-kdd.md`.

## Como usar

- **Rodar uma célula:** clique nela e aperte `Shift + Enter`. Rode de cima para baixo, sem pular.
- **`___` (três sublinhados)** marca o lugar que você completa. Se rodar sem completar, aparece `NameError: name '___' is not defined`. É normal, só quer dizer que falta preencher.
- **Dica:** cada `TODO` tem uma dica no comentário. Se travar, abra a "💡 Dica extra" logo abaixo.
- **Verificação:** depois de cada `TODO` há uma célula que mostra ✅ (certo) ou ❌ (ainda não).
- **✍️ Perguntas:** responda com suas palavras, mesmo que seja um palpite. Errar o palpite e entender por quê é o objetivo.
- **Deu um erro que você não entende?** Copie a mensagem e pergunte ao Claude. Ler mensagens de erro faz parte do aprendizado.

> **Antes de começar:** o ambiente virtual precisa estar pronto (`docs/guia/03-proximos-passos.md`).

## 0. Preparação

Esta célula já está pronta. Ela importa as bibliotecas e cria a função `confere`, usada nas verificações.

In [ ]:
import sys
sys.path.insert(0, "..")  # permite importar o código da pasta src/

import torch
import matplotlib.pyplot as plt
from torchvision import datasets

from src.utils import DATA_DIR, CLASSES


def confere(condicao, mensagem):
    """Mostra ✅ se a condição for verdadeira, ❌ se ainda não."""
    print(("✅ " if condicao else "❌ ") + mensagem)


print("PyTorch:", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())
print("Classes:", CLASSES)

---
# Fase 1 do KDD: Seleção

Escolhemos o **Fashion-MNIST** e usamos a divisão oficial em treino e teste.

A célula abaixo baixa o dataset para a pasta `data/` (só na primeira vez, cerca de 30 MB) e separa:
- `x_train`, `x_test`: as **imagens**;
- `y_train`, `y_test`: os **rótulos**, um número de 0 a 9 que diz a classe de cada imagem.

In [ ]:
treino = datasets.FashionMNIST(DATA_DIR, train=True, download=True)
teste = datasets.FashionMNIST(DATA_DIR, train=False, download=True)

x_train, y_train = treino.data, treino.targets
x_test, y_test = teste.data, teste.targets
print("Pronto!")

### TODO 1: Qual o tamanho dos dados?

Um **tensor** é uma "tabela" de números com várias dimensões. O atributo `.shape` diz o tamanho de cada dimensão.

In [ ]:
# TODO 1: descubra o formato (shape) dos tensores de imagens.
# Dica: escreva o nome do tensor seguido de .shape  (ex.: algum_tensor.shape)
formato_treino = ___
formato_teste = ___

print("Treino:", formato_treino)
print("Teste:", formato_teste)

In [ ]:
confere(tuple(formato_treino) == (60000, 28, 28), "formato do treino")
confere(tuple(formato_teste) == (10000, 28, 28), "formato do teste")

**Pergunta:** o shape do treino tem três números. O que cada um significa?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

### TODO 2: Ver uma imagem e os números por trás dela

In [ ]:
# TODO 2: escolha um número entre 0 e 59999 para ver aquela imagem do treino.
i = ___

plt.imshow(x_train[i], cmap="gray")
plt.title(f"Imagem {i} | classe {y_train[i].item()}: {CLASSES[y_train[i]]}")
plt.axis("off")
plt.show()

Agora veja a mesma imagem como o computador enxerga: uma grade 28x28 de números.

In [ ]:
torch.set_printoptions(linewidth=200)
print(x_train[i])

**Pergunta:** olhando os números, onde está o fundo e onde está a roupa? Que valores aparecem em cada um?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

---
# Fase 2 do KDD: Pré-processamento (qualidade dos dados)

Antes de modelar, conferimos se os dados têm problemas: valores estranhos, classes desbalanceadas ou imagens repetidas.

### TODO 3: Qual a faixa de valores dos pixels?

In [ ]:
# TODO 3: descubra o menor e o maior valor de pixel no treino.
# Dica: tensores têm os métodos .min() e .max()  (ex.: algum_tensor.min())
menor = ___
maior = ___

print("Tipo dos dados:", x_train.dtype)
print("Menor:", menor.item(), "| Maior:", maior.item())

In [ ]:
confere(menor.item() == 0 and maior.item() == 255, "faixa de valores")

**Perguntas:**
1. O tipo é `uint8`. Pesquise rapidamente: que valores um `uint8` pode guardar?
2. Nesta base, o que significa um pixel 0? E um pixel 255?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

### TODO 4: As classes estão balanceadas?

Uma base é **balanceada** quando todas as classes têm quantidades parecidas.

In [ ]:
# TODO 4: conte quantas imagens há em cada classe.
# Dica: torch.bincount(tensor) conta quantas vezes cada número aparece.
#       Qual tensor guarda as classes do treino?
contagem = ___

for c in range(10):
    print(f"{c}  {CLASSES[c]:<12} {contagem[c].item()}")

plt.figure(figsize=(8, 3))
plt.bar(CLASSES, contagem)
plt.xticks(rotation=45, ha="right")
plt.title("Imagens por classe (treino)")
plt.show()

In [ ]:
confere(len(contagem) == 10 and contagem.sum().item() == 60000, "contagem por classe")

**Pergunta:** na Etapa 4, vamos classificar as imagens **geradas** pela GAN e ver quantas caem em cada classe. Considerando o resultado acima, que porcentagem de cada classe uma GAN perfeita deveria gerar? Por quê?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

### TODO 5: Existem imagens repetidas?

Para comparar imagens, cada uma vira uma linha de 784 números (28 x 28). Isso se chama "achatar" (*flatten*).

In [ ]:
planas = x_train.flatten(start_dim=1)    # (60000, 28, 28) -> (60000, 784)
unicas = torch.unique(planas, dim=0)     # remove linhas repetidas

# TODO 5: quantas imagens são duplicatas exatas?
# Dica: compare o total de imagens com o total de imagens únicas. len(tensor) dá o total.
n_duplicadas = ___

print("Total:", len(planas), "| Únicas:", len(unicas), "| Duplicatas:", n_duplicadas)

In [ ]:
confere(n_duplicadas == len(planas) - len(unicas), "contagem de duplicatas")

### TODO 6: Imagens de teste quase iguais a imagens de treino

Para cada uma de 1.000 imagens de teste, procuramos a imagem de treino **mais parecida**. A medida é a **distância L2**: quanto menor, mais parecidas.

Isso vai ser importante na Etapa 4: para saber se a GAN **copiou** o treino, vamos comparar as imagens geradas com essa mesma referência.

A célula abaixo já está pronta. Pode levar alguns segundos.

In [ ]:
amostra_teste = x_test[:1000].flatten(1).float()
treino_plano = x_train.flatten(1).float()

distancias = torch.cdist(amostra_teste, treino_plano)   # (1000, 60000): todas contra todas
dist_min, idx_vizinho = distancias.min(dim=1)           # a menor distância de cada imagem de teste

plt.hist(dist_min, bins=50)
plt.xlabel("distância L2 até a imagem de treino mais parecida")
plt.ylabel("quantidade de imagens de teste")
plt.show()

In [ ]:
ordem = dist_min.argsort()   # índices das imagens de teste, da mais parecida para a menos

# TODO 6: pegue os índices dos 5 pares MAIS parecidos.
# Dica: ordem[:n] pega os n primeiros elementos.
mais_proximos = ___

fig, eixos = plt.subplots(2, 5, figsize=(10, 4))
for col, t in enumerate(mais_proximos):
    eixos[0, col].imshow(x_test[t], cmap="gray")
    eixos[0, col].set_title(f"teste\nd={dist_min[t]:.0f}", fontsize=8)
    eixos[1, col].imshow(x_train[idx_vizinho[t]], cmap="gray")
    eixos[1, col].set_title("treino", fontsize=8)
for ax in eixos.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
confere(len(mais_proximos) == 5, "selecionou 5 pares")

**Perguntas:**
1. Os pares acima são quase iguais? Isso indica algum problema na base?
2. Imagine que a GAN gere uma imagem a uma distância de 50 da imagem de treino mais próxima. Olhando o histograma, isso seria suspeito de cópia ou normal?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

---
# Fase 3 do KDD: Transformação

Redes neurais treinam melhor com números pequenos e centrados. O gerador da GAN termina com a função `tanh`, que só produz valores entre **-1 e 1** (veja `src/models.py`). Por isso as imagens reais precisam estar na mesma faixa.

### TODO 7: Converter de [0, 255] para [-1, 1]

In [ ]:
# TODO 7: transforme x_train para a faixa [-1, 1].
# Dica 1: primeiro converta para número decimal com .float()
# Dica 2: dividir por 127.5 leva [0, 255] para [0, 2]. O que falta para chegar em [-1, 1]?
x_norm = ___

print("Menor:", x_norm.min().item(), "| Maior:", x_norm.max().item())

<details><summary>💡 Dica extra</summary>

Faça a conta com os extremos: 0 / 127.5 = 0 e 255 / 127.5 = 2. Que operação transforma 0 em -1 e 2 em 1?
</details>

In [ ]:
from src.utils import load_fashion_mnist

x_ref, _ = load_fashion_mnist(train=True)   # a versão usada no treino da GAN
confere(abs(x_norm.min().item() + 1) < 1e-6 and abs(x_norm.max().item() - 1) < 1e-6, "faixa [-1, 1]")
confere(torch.allclose(x_norm, x_ref.squeeze(1)), "igual à transformação usada no projeto")

**Pergunta:** o que aconteceria se treinássemos a GAN com imagens em [0, 255], mas o gerador só conseguisse produzir valores em [-1, 1]?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

---
# Explorando padrões

Agora que os dados estão limpos e transformados, vamos entender o que o gerador vai precisar aprender.

### TODO 8: A "roupa típica" de cada classe

A **imagem média** de uma classe mostra o formato típico dela. O **desvio padrão** mostra onde as imagens mais variam entre si.

In [ ]:
medias, desvios = [], []
for c in range(10):
    # TODO 8: selecione só as imagens da classe c.
    # Dica: (y_train == c) cria uma "máscara" de True/False do tamanho do treino.
    #       Colocar essa máscara entre colchetes em x_train filtra as imagens: x_train[mascara]
    imagens_da_classe = ___

    medias.append(imagens_da_classe.float().mean(dim=0))
    desvios.append(imagens_da_classe.float().std(dim=0))

medias = torch.stack(medias)
desvios = torch.stack(desvios)

fig, eixos = plt.subplots(2, 10, figsize=(15, 3.5))
for c in range(10):
    eixos[0, c].imshow(medias[c], cmap="gray")
    eixos[0, c].set_title(CLASSES[c], fontsize=8)
    eixos[1, c].imshow(desvios[c], cmap="magma")
for ax in eixos.flat:
    ax.axis("off")
eixos[0, 0].text(-12, 14, "média", rotation=90, va="center")
eixos[1, 0].text(-12, 14, "desvio", rotation=90, va="center")
plt.show()

In [ ]:
confere(tuple(medias.shape) == (10, 28, 28), "10 imagens médias de 28x28")
confere(len(imagens_da_classe) == 6000, "filtro por classe")

**Perguntas:**
1. Qual imagem média está mais nítida? E qual está mais borrada? O que isso diz sobre a variedade dentro da classe?
2. Qual classe você acha que vai ser **mais difícil** para o gerador aprender? Por quê?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

### TODO 9: Quais classes se parecem entre si?

In [ ]:
# TODO 9: calcule a distância entre as imagens médias de todas as classes.
# Dica: torch.cdist(A, B) calcula a distância de cada linha de A para cada linha de B.
#       Use as médias achatadas (medias.flatten(1)) como A e como B.
dist_classes = ___

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(dist_classes, cmap="viridis")
ax.set_xticks(range(10), CLASSES, rotation=45, ha="right")
ax.set_yticks(range(10), CLASSES)
for a in range(10):
    for b in range(10):
        ax.text(b, a, f"{dist_classes[a, b]:.0f}", ha="center", va="center", fontsize=7, color="white")
plt.colorbar(im, label="distância entre as médias")
plt.title("Quão diferentes são as classes? (menor = mais parecidas)")
plt.show()

In [ ]:
confere(tuple(dist_classes.shape) == (10, 10), "matriz 10x10")

**Perguntas:**
1. Por que a diagonal é toda zero?
2. Quais são os 3 pares de classes **mais parecidos**?
3. **Previsão para a Etapa 4:** quais classes a GAN mais vai misturar, gerando algo "no meio do caminho" entre duas roupas?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

### TODO 10: Projeção 2D com PCA

Cada imagem tem 784 dimensões, impossível de desenhar. O **PCA** (Análise de Componentes Principais) é uma técnica de **aprendizado não supervisionado**: ela encontra as direções em que os dados mais variam e permite "achatar" as 784 dimensões em 2, perdendo o mínimo de informação.

Repare que o PCA **não usa os rótulos**. As cores no gráfico são só para nós conferirmos o resultado.

In [ ]:
from sklearn.decomposition import PCA

n = 5000
X = x_train[:n].flatten(1).float().numpy() / 255   # 5000 imagens, cada uma com 784 números
rotulos = y_train[:n].numpy()

# TODO 10: crie um PCA com 2 componentes e aplique em X.
# Dica: pca = PCA(n_components=...)   e depois   X_2d = pca.fit_transform(...)
pca = ___
X_2d = ___

plt.figure(figsize=(8, 7))
for c in range(10):
    pontos = X_2d[rotulos == c]
    plt.scatter(pontos[:, 0], pontos[:, 1], s=3, alpha=0.5, label=CLASSES[c])
plt.legend(markerscale=5, fontsize=8)
plt.title("Fashion-MNIST projetado em 2D (PCA)")
plt.show()

print(f"Variância explicada pelos 2 componentes: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
confere(X_2d.shape == (5000, 2), "projeção em 2D")

**Perguntas:**
1. Quais classes formam grupos separados? Quais ficam misturadas?
2. Isso confirma o que você viu na matriz de distâncias do TODO 9?
3. Os 2 componentes explicam qual porcentagem da variação? O que isso diz sobre o quanto de informação se perde no gráfico?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

### 🏆 Desafio opcional: t-SNE

O **t-SNE** é outra técnica de projeção, não linear, que costuma separar melhor os grupos. Ela é mais lenta (cerca de 1 minuto). Tente escrever sozinho, adaptando o código do PCA:

- importe com `from sklearn.manifold import TSNE`;
- use `TSNE(n_components=2, random_state=42).fit_transform(X[:3000])`;
- lembre de usar `rotulos[:3000]` nas cores.

In [ ]:
# Desafio: seu código do t-SNE aqui

---
# Contexto dos dados (para a análise ética)

O Fashion-MNIST foi criado pela **Zalando**, uma loja online europeia, a partir de fotos do seu catálogo de produtos. Pesquise rapidamente e responda:

1. Quem escolheu quais roupas entraram na base e quais ficaram de fora?
2. Que tipos de roupa ou de estilo provavelmente **não** estão representados?
3. Se uma GAN treinada aqui fosse usada para "gerar roupas", que viés ela herdaria?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

---
# Resumo: o que descobri

Preencha a tabela. Depois, peça ao Claude para registrar isso em `docs/experimentos/`, porque essas conclusões vão para o relatório.

| Pergunta | O que descobri |
|---|---|
| Tamanho da base (treino / teste) | |
| Faixa de valores original | |
| Classes balanceadas? | |
| Duplicatas exatas no treino | |
| Distância típica teste → treino | |
| Classe mais variada | |
| Pares de classes mais parecidos | |
| Previsão: onde a GAN vai errar | |
| Possíveis vieses da base | |